In [ ]:
import os
from openai import OpenAI
import threading
from dotenv import load_dotenv

In [ ]:
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [ ]:
def call_openai(system_prompt , user_prompt):
  response = client.chat.completions(
      model="gpt-3.5-turbo",
      messages=[
          {"role":"system","content":syetem_prompt},
          {"role":"user","content":user_promp}
      ],
      temprature =0,
  )
  return response.choice[0].message.content
  Max_RETRIES = 3

In [ ]:
class RecipeCreatorAgent:
  def create_recipe(self, recipe_request_dict,feedback=None):
     system_message = """You are a creative chef... Experiment... try something bold, don't be afraid to break the rules.""" # Initial creative prompt

     if feedback is None:
         user_prompt = f"""Recipe Request: {recipe_request}
         Create an exciting, flavorful recipe... prioritize creativity and taste first."""
         print("\n Creating initial recipe (flexible interpretation)...")
         current_temperature = 1
     else:
         system_message = """You are an expert chef specializing in creating recipes that follow strict dietary constraints.
         You must correct previous issues and follow all requirements with precision."""
         user_prompt = f"""Recipe Request: {recipe_request}
         Your previous recipe had the following issues:\n{feedback}\nPlease create a revised recipe addressing these issues."""
         print("\n Generating revised recipe based on feedback...")
         current_temperature = 0.3 # Lower for more precision

     recipe = call_openai(system_message,user_prompt)
     return recipe


In [ ]:
class NutritionEvaluatorAgent:
  def evaluate(self , recipe_details_str, original_request_dict):
    system_message="""You are an extremely precise nutrition and dietary compliance evaluator. Your role is to meticulously assess a given recipe against a specific set of user-defined constraints. For each constraint, you must clearly state if it 'PASSED' or 'FAILED'. If a constraint FAILED, you must provide a concise reason and a actionable suggestion for improvement. You also need to provide an overall taste rating based on the recipe description."""
    user_prompt= f"""Please evaluate the RECIPE above against EACH of the following constraints from the original REQUEST:
    Original Request Constraints: {', '.join(original_request_dict['constraints'])}
    For each constraint, state the constraint verbatim, then write 'PASSED' or 'FAILED'.
    If 'FAILED', provide a brief reason and a specific suggestion for fixing it.
    Example for one constraint:
    'gluten-free: PASSED'
    'under 500 calories per serving: FAILED - Estimated 650 calories. Suggest reducing oil by half.'
    After evaluating all constraints, provide a line with 'Taste Rating: [N]/10' based on the recipe description (where N is a number).
    Finally, on a new line, write 'Overall Status: PASSED' if ALL constraints are met (including taste rating of 7 or higher for the constraint 'taste must be rated 7/10 or higher') OR 'Overall Status: FAILED' if ANY constraint is not met."""
    return call_openai(system_prompt,user_prompt)

In [ ]:
def optimize_recipe(recipe_request):
    creator = RecipeCreatorAgent()
    evaluator = NutritionEvaluatorAgent()
    feedback = None
    attempts = 0

    for attempt in range(MAX_RETRIES): # Iterative Loop
        attempts += 1
        print(f"\n--- Attempt #{attempts} ---")

        recipe = creator.create_recipe(recipe_request, feedback) # Generation/Revision
        evaluation = evaluator.evaluate(recipe_request, recipe)  # Evaluation

        print(f"\n📋 Evaluation Result:\n{evaluation[:200] + "..." if len(evaluation) > 200 else evaluation}")

        if evaluation.lower().startswith("approved"): # Stopping Condition 1
            print("\n✅ All dietary constraints satisfied!")
            break # Exit loop on success
        else:
            feedback = evaluation # Feedback Propagation for next iteration
            print(f"\n⚠️ Some constraints not met. Optimizing recipe...")

    # After loop (either approved or max retries hit)
    return recipe, evaluation, attempts


In [ ]:
recipe = "chocolate cake"
optimize_recipe(recipe)